In [2]:
import numpy as np
import torch

In [3]:
with open(r"C:\Users\Hp\OneDrive\Desktop\myprog\ai\dataset\input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Characters:", len(text))
print("Unique characters:", len(set(text)))

Characters: 1115394
Unique characters: 65


In [4]:
text[:50]

'First Citizen:\nBefore we proceed any further, hear'

In [5]:
chars=sorted(list(set(text)))
vocab_size=len(chars)
print(vocab_size)
print("".join(chars))

65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [6]:
stoi= {ch:i for i, ch in enumerate(chars)} 
itos= {i:ch for i, ch in enumerate(chars)} 

encode= lambda s: [stoi[c] for c in s] 
decode= lambda s: "".join([itos[c] for c in s]) 

print(encode("hii i am ak") )
print(decode(encode("hii i am ak")))

[46, 47, 47, 1, 47, 1, 39, 51, 1, 39, 49]
hii i am ak


In [7]:
import torch
data=torch.tensor(encode(text), dtype=torch.long)
print(data.shape)
print(data[:40])

torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56])


In [8]:
n=int(0.9*len(text))
train_data=data[:n]
val_data=data[n:]

In [9]:
batch_size=4
block_size=8
torch.manual_seed(1337)


def get_batch(split):
    data= train_data if split=="train" else val_data

    ix=torch.randint( len(data) - block_size, (batch_size,) )
    x=torch.stack( [data[i: i+block_size] for i in ix ] ) 
    y=torch.stack( [data[i+1: i+block_size+1] for i in ix ] )

    return x,y

xb,yb =get_batch("train")

print("inputs" ,xb)
print("targets" ,yb)
print(xb.shape)

for b in range(batch_size):
    for t in range(block_size):
        print(f" input is {xb[b,:t+1].tolist()} and output is {yb[b, t]}")


inputs tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
torch.Size([4, 8])
 input is [24] and output is 43
 input is [24, 43] and output is 58
 input is [24, 43, 58] and output is 5
 input is [24, 43, 58, 5] and output is 57
 input is [24, 43, 58, 5, 57] and output is 1
 input is [24, 43, 58, 5, 57, 1] and output is 46
 input is [24, 43, 58, 5, 57, 1, 46] and output is 43
 input is [24, 43, 58, 5, 57, 1, 46, 43] and output is 39
 input is [44] and output is 53
 input is [44, 53] and output is 56
 input is [44, 53, 56] and output is 1
 input is [44, 53, 56, 1] and output is 58
 input is [44, 53, 56, 1, 58] and output is 46
 input is [44, 53, 56, 1, 58, 46] and output is 39
 input is [44, 53, 56,

In [10]:
batch_size = 16
block_size = 32
n_head = 4
n_layer = 4
dropout = 0.0
n_embd=64

In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)
class Head(nn.Module):
    def __init__(self, n_embd,head_size):
        super().__init__()
        self.key=nn.Linear(n_embd, head_size, bias=False)
        self.query=nn.Linear(n_embd, head_size, bias=False)
        self.value=nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer( "tril" , torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    
    def forward(self,x):
        B,T,C=x.shape
        k=self.key(x)
        q=self.query(x)
        v=self.value(x)

        wei= q @ k.transpose(-2, -1) *(k.shape[-1]** -0.5)

        wei=wei.masked_fill(self.tril[:T, :T]==0, float("-inf"))
        wei=F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out=wei @ v
        return out


        


In [12]:
class MultiHead(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size= n_embd // n_head
        self.heads=nn.ModuleList([ Head(n_embd,head_size) for _ in range(n_head) ])
        self.projection=nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        out=torch.cat( [h(x) for h in self.heads], dim=-1 )
        out=self.projection(out)
        out = self.dropout(out)
        return out
        


In [13]:
class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout),

        )

    def forward(self, x):
        return self.net(x)
        

In [14]:
class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        self.multi_head=MultiHead(n_embd, n_head)
        self.feed_forward=FeedForward(n_embd)
        self.ln1=nn.LayerNorm(n_embd)
        self.ln2=nn.LayerNorm(n_embd)
        
    
    def forward(self, x):
        x = x + self.multi_head(self.ln1(x))
        x = x + self.feed_forward(self.ln2(x))
        return x

In [15]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)


class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size, n_embd, n_head, n_layer):
        super().__init__()
        self.token_embedding_table=nn.Embedding(vocab_size,n_embd )
        self.position_embedding_table=nn.Embedding(block_size, n_embd)
        self.layer_norm=nn.LayerNorm(n_embd)
        self.blocks=nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])

        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B,T=idx.shape
        tok_emd=self.token_embedding_table(idx)
        pos_emd = self.position_embedding_table(torch.arange(T))
        x=tok_emd +pos_emd
        x=self.blocks(x)
        x=self.layer_norm(x)
        logits=self.lm_head(x)

        
        if targets is None:
            loss=None
        else:
            B,T,C=logits.shape
            logits= logits.view(B*T, C)
            targets= targets.view(B*T)
            loss=F.cross_entropy(logits, targets)

        return logits,loss

    def generate(self, idx, max_tokens):

        for _ in range(max_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss=self(idx_cond) # idx=b,t logits=b,t,c

            logits= logits[ :, -1,: ] # logits= b,c
            probs=F.softmax(logits, dim=-1)

            idx_next=torch.multinomial(probs, num_samples=1) #b,1

            idx=torch.cat((idx,idx_next), dim=1) # to b,t+1
        return idx





In [16]:


m=BigramLanguageModel(vocab_size, n_embd, n_head, n_layer)
logits, loss=m(xb, yb)
print(logits.shape)
print(loss)
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_tokens=100)[0].tolist()))


torch.Size([32, 65])
tensor(4.5082, grad_fn=<NllLossBackward0>)

DuJAMJ3fX$VWoGCYngj$'onoE,$AT-NvynticCzJ-c!bnDNyjwND,?to;
VcANzq:nU$3NVAz:YzgNkceei.-efVhpXz,NApQe.3


In [17]:
optimizer=torch.optim.AdamW(m.parameters(), lr=1e-3)

In [18]:
eval_iters=200
@torch.no_grad()
def estimate_loss():
    out = {}
    m.eval()

    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = m(X, Y)
            losses[k] = loss.item()

        out[split] = losses.mean()

    m.train()
    return out

In [19]:
epochs=5000
for epoch in range(epochs):

    if epoch % 100 == 0:
        losses = estimate_loss()
        print(
            f"step {epoch}: "
            f"train loss {losses['train']:.4f}, "
            f"val loss {losses['val']:.4f}"
        )

    xb, yb = get_batch("train")

    logits, loss = m(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.4078, val loss 4.4023
step 100: train loss 2.6406, val loss 2.6444
step 200: train loss 2.4899, val loss 2.4877
step 300: train loss 2.3817, val loss 2.3786
step 400: train loss 2.3070, val loss 2.3151
step 500: train loss 2.2538, val loss 2.2500
step 600: train loss 2.2054, val loss 2.2135
step 700: train loss 2.1614, val loss 2.1702
step 800: train loss 2.1089, val loss 2.1513
step 900: train loss 2.0863, val loss 2.1223
step 1000: train loss 2.0609, val loss 2.1001
step 1100: train loss 2.0352, val loss 2.0919
step 1200: train loss 2.0004, val loss 2.0531
step 1300: train loss 1.9815, val loss 2.0362
step 1400: train loss 1.9477, val loss 2.0210
step 1500: train loss 1.9385, val loss 2.0061
step 1600: train loss 1.9242, val loss 2.0016
step 1700: train loss 1.9054, val loss 1.9899
step 1800: train loss 1.8853, val loss 1.9605
step 1900: train loss 1.8635, val loss 1.9626
step 2000: train loss 1.8568, val loss 1.9686
step 2100: train loss 1.8494, val loss 1.9481


In [31]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_tokens=1000)[0].tolist()))



As godood Nomber
Warwice thy bore prolomp him then that self.

CLARES:
Alatter but we note, an marrialy-cands
Which mistin, shillow Loss of you shill
what all not mastly end thine 'ell men men him; he bet his mine; by find eart bloudy's my hown;
Were bed dotephy pries, nect all ry balm
O, as subhiles licer whee
mon were in my pleastes, well shall? it, sir, by thou I wargun:
Eilt aripos too being
The will hall were before ship trome.

HENRMIOLANE:
Os her studdes, the her desenem
Be be a ast Hends. Lond, of an libbate a mirisude to you'll bour reall child,
The wilt'st dones an thy heares ush-on,
And tuo, of this his strefend in lawn's law,
The heal of you. A contenict this dead; you hass, be you the earts, there, arother's could my Gites at bilick
No.
What where 'Fzose your shy; and time sea-Beass deeps mase.
You its nother I slubjeed, till thy make strecort'st thing be in Cly
As of me is by uld naturiblia!
On--dow; then-make WI'll arm, he parest,
And thus kep'd than nobler,
The beatt; 

In [21]:
print(sum(p.numel() for p in m.parameters()))

209729


In [22]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [23]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [24]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)


In [25]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
# print(wei)
wei = wei / wei.sum(1, keepdim=True)
# print(wei)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)
torch.allclose(xbow, xbow2, atol=1e-6)

True

In [26]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
# print(tril)
wei = torch.zeros((T,T))
# print(wei)
wei = wei.masked_fill(tril == 0, float('-inf'))
# print(wei)
wei = F.softmax(wei, dim=-1)
# print(wei)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)


False

In [27]:
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

key=nn.Linear(C, 32, bias=False)
query=nn.Linear(C, 32, bias=False)
value=nn.Linear(C, 32, bias=False)

k=key(x)
q=query(x)
v=value(x)

wei= q@ k.transpose(-2, -1)

tril=torch.tril(torch.ones(T,T))
wei=wei.masked_fill(tril==0, float("-inf") )
wei=F.softmax(wei, dim=-1)

out=wei @ v





In [28]:
x.shape

torch.Size([4, 8, 32])